# Model Selection Cache Recovery

Recovered workflow. See `docs/WORKFLOW_ORDER.md` for dependencies and execution restrictions. Source cell numbers below are zero-based.


Source cell: 86.


In [ ]:
# ============================================================
# REBUILD TRAIN-ONLY FEATURE CACHE CORRECTLY
#
# IMPORTANT:
# Drug1 and Drug2 ARE SMILES.
#
# NO MODEL TRAINING.
# ============================================================

import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors


# ============================================================
# 1) PATHS
# ============================================================

ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/ddi-v2-menna-projectt"
)

TRAIN_FILE = ROOT / "clean_train_split.csv"
CAL_FILE   = ROOT / "clean_calibration_split.csv"
TEST_FILE  = ROOT / "clean_test_split.csv"

OLD_CACHE = ROOT / "exact_v2_feature_cache_2249.npz"
OLD_META  = ROOT / "exact_v2_feature_meta.json"

OUT = Path(
    "/kaggle/working/"
    "FEATURE_ABLATION_TRAIN_ONLY_PREPROCESSING"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


for p in [
    TRAIN_FILE,
    CAL_FILE,
    TEST_FILE,
    OLD_CACHE,
    OLD_META,
]:
    assert p.is_file(), f"Missing: {p}"


# ============================================================
# 2) LOAD SPLITS
# ============================================================

train_df = pd.read_csv(TRAIN_FILE)
cal_df   = pd.read_csv(CAL_FILE)
test_df  = pd.read_csv(TEST_FILE)

assert len(train_df) == 134249
assert len(cal_df) == 28768
assert len(test_df) == 28768

required = {
    "Drug1",
    "Drug2",
    "Label",
    "label_id",
}

for name, df in [
    ("Train", train_df),
    ("Calibration", cal_df),
    ("Test", test_df),
]:
    assert required.issubset(df.columns), (
        name,
        df.columns.tolist()
    )


print("=" * 90)
print("SPLITS VERIFIED")
print("=" * 90)

print("Train:", train_df.shape)
print("Calibration:", cal_df.shape)
print("Test:", test_df.shape)

print("\n Drug1 and Drug2 are treated directly as SMILES.")


# ============================================================
# 3) GET ALL UNIQUE MOLECULES
# ============================================================

all_split_smiles = set(
    pd.concat(
        [
            train_df["Drug1"].astype(str),
            train_df["Drug2"].astype(str),
            cal_df["Drug1"].astype(str),
            cal_df["Drug2"].astype(str),
            test_df["Drug1"].astype(str),
            test_df["Drug2"].astype(str),
        ],
        ignore_index=True
    ).tolist()
)

assert len(all_split_smiles) == 1705, (
    "Expected 1705 unique molecular SMILES, "
    f"found {len(all_split_smiles)}"
)


train_smiles_set = set(
    train_df["Drug1"].astype(str)
).union(
    set(
        train_df["Drug2"].astype(str)
    )
)

cal_smiles_set = set(
    cal_df["Drug1"].astype(str)
).union(
    set(
        cal_df["Drug2"].astype(str)
    )
)

test_smiles_set = set(
    test_df["Drug1"].astype(str)
).union(
    set(
        test_df["Drug2"].astype(str)
    )
)


assert len(train_smiles_set) == 1675

cal_unseen = (
    cal_smiles_set
    -
    train_smiles_set
)

test_unseen = (
    test_smiles_set
    -
    train_smiles_set
)

assert len(cal_unseen) == 14
assert len(test_unseen) == 20


print("\nMOLECULAR SET QA")
print("Total molecules:", len(all_split_smiles))
print("Train molecules:", len(train_smiles_set))
print("Calibration unseen:", len(cal_unseen))
print("Test unseen:", len(test_unseen))


# ============================================================
# 4) RECOVER HISTORICAL 1705-MOLECULE ORDER
#
# We use OLD CACHE ONLY FOR ORDERING.
# We DO NOT reuse old feature values.
# ============================================================

with np.load(
    OLD_CACHE,
    allow_pickle=True
) as old:

    print("\nHistorical cache keys:")
    print(old.files)

    order_candidates = []

    for key in old.files:

        arr = old[key]

        if arr.ndim != 1:
            continue

        if len(arr) != 1705:
            continue

        try:
            values = arr.astype(str)
        except Exception:
            continue

        if len(np.unique(values)) != 1705:
            continue

        if set(values.tolist()) == all_split_smiles:

            order_candidates.append(
                (
                    key,
                    values.copy()
                )
            )


assert order_candidates, (
    "Could not recover historical molecular order "
    "from exact_v2_feature_cache_2249.npz."
)


print("\nMatching 1705-molecule vectors:")

for key, _ in order_candidates:
    print("  ", key)


all_drugs_key, all_drugs = (
    order_candidates[0]
)


assert len(all_drugs) == 1705
assert set(all_drugs.tolist()) == all_split_smiles


print(
    "\n Historical molecular order recovered from:",
    all_drugs_key
)


# ============================================================
# 5) VALIDATE ALL 1705 SMILES
# ============================================================

mols = []

invalid = []

for i, smi in enumerate(
    all_drugs
):

    mol = Chem.MolFromSmiles(
        str(smi)
    )

    if mol is None:

        invalid.append(
            (
                i,
                smi
            )
        )

    mols.append(
        mol
    )


assert not invalid, (
    f"Invalid SMILES found: {invalid[:10]}"
)


print(
    " All 1705 historical molecules parsed by RDKit."
)


# ============================================================
# 6) TRAIN MOLECULE INDICES
# ============================================================

drug_to_index = {
    str(smi): idx
    for idx, smi
    in enumerate(all_drugs)
}


train_drug_indices = np.array(
    sorted(
        drug_to_index[smi]
        for smi in train_smiles_set
    ),
    dtype=np.int64
)


assert len(train_drug_indices) == 1675


# ============================================================
# 7) RECOVER CANONICAL 208 RDKIT DESCRIPTOR NAMES
# ============================================================

meta = json.loads(
    OLD_META.read_text(
        encoding="utf-8"
    )
)


available_descriptor_names = {
    name
    for name, _
    in Descriptors.descList
}


def search_208_descriptor_lists(obj):

    results = []

    if isinstance(obj, dict):

        for value in obj.values():

            results.extend(
                search_208_descriptor_lists(
                    value
                )
            )

    elif isinstance(obj, list):

        if (
            len(obj) == 208
            and
            all(
                isinstance(x, str)
                for x in obj
            )
        ):

            overlap = sum(
                x in available_descriptor_names
                for x in obj
            )

            if overlap >= 190:

                results.append(
                    obj
                )

        for value in obj:

            results.extend(
                search_208_descriptor_lists(
                    value
                )
            )

    return results


descriptor_candidates = (
    search_208_descriptor_lists(
        meta
    )
)


assert descriptor_candidates, (
    "Could not recover canonical 208 RDKit descriptor list."
)


descriptor_names = (
    descriptor_candidates[0]
)


assert len(descriptor_names) == 208


missing_functions = [
    name
    for name in descriptor_names
    if name not in available_descriptor_names
]


assert not missing_functions, (
    f"Current RDKit is missing: {missing_functions}"
)


print(
    "\nCanonical RDKit descriptors:",
    len(descriptor_names)
)


# ============================================================
# 8) MORGAN 2048
# ============================================================

X_morgan = np.zeros(
    (
        1705,
        2048
    ),
    dtype=np.float32
)


for i, mol in enumerate(mols):

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=2048,
        useChirality=False,
        useFeatures=False,
    )

    arr = np.zeros(
        2048,
        dtype=np.int8
    )

    DataStructs.ConvertToNumpyArray(
        fp,
        arr
    )

    X_morgan[i] = arr


assert X_morgan.shape == (
    1705,
    2048
)

assert np.isfinite(
    X_morgan
).all()


print(
    "\nRaw Morgan shape:",
    X_morgan.shape
)


# ============================================================
# 9) RAW RDKIT 208
# ============================================================

descriptor_fn = dict(
    Descriptors.descList
)


X_rdkit_raw = np.full(
    (
        1705,
        208
    ),
    np.nan,
    dtype=np.float64
)


for i, mol in enumerate(mols):

    for j, name in enumerate(
        descriptor_names
    ):

        try:

            value = descriptor_fn[
                name
            ](
                mol
            )

            if np.isfinite(value):

                X_rdkit_raw[
                    i,
                    j
                ] = float(value)

        except Exception:

            pass


nonfinite_count = int(
    (
        ~np.isfinite(
            X_rdkit_raw
        )
    ).sum()
)


print(
    "Raw RDKit shape:",
    X_rdkit_raw.shape
)

print(
    "Raw RDKit non-finite values:",
    nonfinite_count
)


# Previous completed Cell A produced 618.
assert nonfinite_count == 618, (
    "RDKit descriptor calculation differs from "
    "the completed feature-ablation preprocessing.\n"
    f"Expected 618 non-finite values, got {nonfinite_count}."
)


# ============================================================
# 10) TRAIN-ONLY MEDIAN + VARIANCE SELECTION
# ============================================================

train_raw = (
    X_rdkit_raw[
        train_drug_indices
    ]
)


train_medians = np.zeros(
    208,
    dtype=np.float64
)

keep_mask = np.zeros(
    208,
    dtype=bool
)


for j in range(208):

    col = train_raw[:, j]

    finite = col[
        np.isfinite(
            col
        )
    ]

    if len(finite) == 0:

        median = 0.0

        train_medians[j] = median

        keep_mask[j] = False

        continue


    median = float(
        np.median(
            finite
        )
    )


    train_medians[j] = median


    imputed_train_col = np.where(
        np.isfinite(col),
        col,
        median,
    )


    variance = float(
        np.var(
            imputed_train_col
        )
    )


    keep_mask[j] = (
        variance > 1e-12
    )


retained_names = [
    name
    for name, keep
    in zip(
        descriptor_names,
        keep_mask
    )
    if keep
]


dropped_names = [
    name
    for name, keep
    in zip(
        descriptor_names,
        keep_mask
    )
    if not keep
]


expected_dropped = [
    "Ipc",
    "SMR_VSA8",
    "SlogP_VSA9",
    "fr_diazo",
    "fr_isocyan",
    "fr_isothiocyan",
    "fr_thiocyan",
]


assert len(retained_names) == 201

assert set(
    dropped_names
) == set(
    expected_dropped
), (
    "\nTrain-only descriptor selection differs "
    "from previous controlled preprocessing.\n\n"
    f"Actual dropped:\n{dropped_names}"
)


print("\n" + "=" * 90)
print("TRAIN-ONLY RDKIT FIT")
print("=" * 90)

print("Original:", 208)
print("Retained:", len(retained_names))
print("Dropped:", len(dropped_names))
print("Dropped names:", dropped_names)


# ============================================================
# 11) IMPUTE ALL MOLECULES USING TRAIN MEDIANS
# ============================================================

X_rdkit_imputed = (
    X_rdkit_raw.copy()
)


for j in range(208):

    bad = ~np.isfinite(
        X_rdkit_imputed[:, j]
    )

    X_rdkit_imputed[
        bad,
        j
    ] = train_medians[j]


X_rdkit_selected = (
    X_rdkit_imputed[
        :,
        keep_mask
    ]
)


assert X_rdkit_selected.shape == (
    1705,
    201
)


# ============================================================
# 12) TRAIN-ONLY STANDARD SCALER
# ============================================================

scaler = StandardScaler()


scaler.fit(
    X_rdkit_selected[
        train_drug_indices
    ]
)


X_rdkit = (
    scaler.transform(
        X_rdkit_selected
    )
    .astype(
        np.float32
    )
)


assert X_rdkit.shape == (
    1705,
    201
)

assert np.isfinite(
    X_rdkit
).all()


# ============================================================
# 13) COMBINED REPRESENTATION
# ============================================================

X_combined = np.concatenate(
    [
        X_morgan,
        X_rdkit,
    ],
    axis=1
).astype(
    np.float32
)


assert X_combined.shape == (
    1705,
    2249
)

assert np.isfinite(
    X_combined
).all()


print("\n" + "=" * 90)
print("FINAL FEATURE MATRICES")
print("=" * 90)

print(
    "Morgan-only:",
    X_morgan.shape
)

print(
    "RDKit-only:",
    X_rdkit.shape
)

print(
    "Morgan + RDKit:",
    X_combined.shape
)


# ============================================================
# 14) SAVE ALL THREE CACHES
# ============================================================

MORGAN_FILE = (
    OUT /
    "molecular_features_morgan_only.npz"
)

RDKIT_FILE = (
    OUT /
    "molecular_features_rdkit_only.npz"
)

COMBINED_FILE = (
    OUT /
    "molecular_features_morgan_rdkit.npz"
)


common_arrays = {
    "all_drugs":
        np.asarray(
            all_drugs
        ),

    "train_drug_indices":
        train_drug_indices,
}


np.savez_compressed(
    MORGAN_FILE,
    x=X_morgan,
    **common_arrays,
)


np.savez_compressed(
    RDKIT_FILE,
    x=X_rdkit,
    **common_arrays,
)


np.savez_compressed(
    COMBINED_FILE,
    x=X_combined,
    **common_arrays,
)


# ============================================================
# 15) METADATA
# ============================================================

metadata = {

    "experiment":
        "controlled_feature_ablation_train_only",

    "identity_definition":
        "Drug1/Drug2 columns are molecular SMILES",

    "historical_order_source":
        str(OLD_CACHE),

    "historical_order_key":
        all_drugs_key,

    "old_feature_values_reused":
        False,

    "total_drugs":
        1705,

    "train_fit_drugs":
        1675,

    "calibration_unseen_drugs":
        14,

    "test_unseen_drugs":
        20,

    "morgan_radius":
        2,

    "morgan_bits":
        2048,

    "morgan_use_chirality":
        False,

    "morgan_use_features":
        False,

    "rdkit_raw_count":
        208,

    "rdkit_nonfinite_raw":
        nonfinite_count,

    "rdkit_retained_train_only":
        201,

    "rdkit_dropped_train_only":
        dropped_names,

    "rdkit_retained_names":
        retained_names,

    "imputation":
        "Train-molecule median per descriptor",

    "selection":
        "Train variance after imputation > 1e-12",

    "scaling":
        "StandardScaler fit on Train molecules only",

    "morgan_dim":
        2048,

    "rdkit_dim":
        201,

    "combined_dim":
        2249,
}


META_FILE = (
    OUT /
    "feature_ablation_metadata.json"
)


META_FILE.write_text(
    json.dumps(
        metadata,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 16) FINAL RELOAD QA
# ============================================================

for path, expected_shape in [

    (
        MORGAN_FILE,
        (1705, 2048)
    ),

    (
        RDKIT_FILE,
        (1705, 201)
    ),

    (
        COMBINED_FILE,
        (1705, 2249)
    ),
]:

    with np.load(
        path,
        allow_pickle=True
    ) as check:

        assert (
            check["x"].shape
            ==
            expected_shape
        )

        assert len(
            check["all_drugs"]
        ) == 1705

        assert len(
            check["train_drug_indices"]
        ) == 1675

        assert np.isfinite(
            check["x"]
        ).all()


# ============================================================
# 17) ZIP PREPROCESSING ARTIFACT
# ============================================================

ZIP_PATH = Path(
    "/kaggle/working/"
    "FEATURE_ABLATION_TRAIN_ONLY_PREPROCESSING_RECOVERED.zip"
)


with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as z:

    for p in [
        MORGAN_FILE,
        RDKIT_FILE,
        COMBINED_FILE,
        META_FILE,
    ]:

        z.write(
            p,
            arcname=p.name
        )


with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as z:

    assert z.testzip() is None


# ============================================================
# 18) FINAL REPORT
# ============================================================

print("\n" + "=" * 90)
print(" TRAIN-ONLY FEATURE PREPROCESSING RECOVERED")
print("=" * 90)

print("Total molecular SMILES:", 1705)
print("Train-fit molecules:", 1675)
print("CAL unseen:", 14)
print("TEST unseen:", 20)

print("\nRDKit:")
print("Canonical descriptors:", 208)
print("Raw non-finite:", nonfinite_count)
print("Retained:", 201)
print("Dropped:", dropped_names)

print("\nShapes:")
print("Morgan:", X_morgan.shape)
print("RDKit:", X_rdkit.shape)
print("Combined:", X_combined.shape)

print("\nCombined cache:")
print(COMBINED_FILE)

print("\nZIP:")
print(ZIP_PATH)

print("\n Historical order only reused")
print(" Historical feature values NOT reused")
print(" Train-only descriptor selection")
print(" Train-only median imputation")
print(" Train-only scaling")
print(" No model training performed")

print(
    "\nAfter this cell completes, "
    "run the model-selection sensitivity cell from the start."
)

Source cell: 87.


In [ ]:
# ============================================================
# FIX RDKIT TRAIN-ONLY VARIANCE SELECTION
# CONTINUE FROM CURRENT SESSION
#
# Requires these variables already in memory:
# X_morgan
# X_rdkit_raw
# train_drug_indices
# descriptor_names
# all_drugs
# OUT
#
# NO MODEL TRAINING
# ============================================================

import json
import zipfile
from pathlib import Path

import numpy as np
from sklearn.preprocessing import StandardScaler


# ============================================================
# 1) BASIC STATE QA
# ============================================================

assert X_morgan.shape == (1705, 2048)
assert X_rdkit_raw.shape == (1705, 208)
assert len(train_drug_indices) == 1675
assert len(descriptor_names) == 208
assert len(all_drugs) == 1705

assert int(
    (~np.isfinite(X_rdkit_raw)).sum()
) == 618


print("=" * 90)
print("FIXING TRAIN-ONLY DESCRIPTOR SELECTION")
print("=" * 90)


# ============================================================
# 2) TRAIN-ONLY MEDIAN + FINITE VARIANCE SELECTION
# ============================================================

train_raw = X_rdkit_raw[
    train_drug_indices
].copy()


train_medians = np.zeros(
    208,
    dtype=np.float64
)

train_variances = np.full(
    208,
    np.nan,
    dtype=np.float64
)

keep_mask = np.zeros(
    208,
    dtype=bool
)


for j in range(208):

    col = train_raw[:, j]

    finite_values = col[
        np.isfinite(col)
    ]


    # Completely unusable on Train
    if len(finite_values) == 0:

        train_medians[j] = 0.0
        train_variances[j] = np.nan
        keep_mask[j] = False
        continue


    median = float(
        np.median(
            finite_values
        )
    )

    train_medians[j] = median


    imputed_col = np.where(
        np.isfinite(col),
        col,
        median
    )


    # Important:
    # calculate in float64 but explicitly reject inf/nan variance
    with np.errstate(
        over="ignore",
        invalid="ignore"
    ):

        variance = float(
            np.var(
                imputed_col,
                dtype=np.float64
            )
        )


    train_variances[j] = variance


    keep_mask[j] = (
        np.isfinite(variance)
        and
        variance > 1e-12
    )


# ============================================================
# 3) REPORT EXACT DROPPED DESCRIPTORS
# ============================================================

retained_names = [
    name
    for name, keep
    in zip(
        descriptor_names,
        keep_mask
    )
    if keep
]

dropped_names = [
    name
    for name, keep
    in zip(
        descriptor_names,
        keep_mask
    )
    if not keep
]


expected_dropped = {
    "Ipc",
    "SMR_VSA8",
    "SlogP_VSA9",
    "fr_diazo",
    "fr_isocyan",
    "fr_isothiocyan",
    "fr_thiocyan",
}


print(
    "Retained descriptors:",
    len(retained_names)
)

print(
    "Dropped descriptors:",
    len(dropped_names)
)

print(
    "Dropped names:",
    dropped_names
)


print("\nDropped descriptor diagnostics:")

for name in dropped_names:

    j = descriptor_names.index(name)

    print(
        f"{name:20s} "
        f"variance={train_variances[j]} "
        f"finite_train="
        f"{np.isfinite(train_raw[:, j]).sum()}/{len(train_raw)}"
    )


# ============================================================
# 4) STRICT MATCH TO SUCCESSFUL CELL A
# ============================================================

assert len(retained_names) == 201, (
    f"Expected 201 retained descriptors, "
    f"found {len(retained_names)}.\n"
    f"Dropped: {dropped_names}"
)

assert set(dropped_names) == expected_dropped, (
    "Dropped descriptor set does NOT match "
    "the successful controlled Cell A.\n\n"
    f"Expected:\n{sorted(expected_dropped)}\n\n"
    f"Actual:\n{sorted(dropped_names)}"
)


print(
    "\n Exact original Train-only selection recovered."
)


# ============================================================
# 5) IMPUTE ALL 1705 USING TRAIN MEDIANS
# ============================================================

X_rdkit_imputed = X_rdkit_raw.copy()


for j in range(208):

    bad = ~np.isfinite(
        X_rdkit_imputed[:, j]
    )

    X_rdkit_imputed[
        bad,
        j
    ] = train_medians[j]


X_rdkit_selected = X_rdkit_imputed[
    :,
    keep_mask
]


assert X_rdkit_selected.shape == (
    1705,
    201
)

assert np.isfinite(
    X_rdkit_selected
).all()


# ============================================================
# 6) TRAIN-ONLY STANDARD SCALER
# ============================================================

scaler = StandardScaler()


scaler.fit(
    X_rdkit_selected[
        train_drug_indices
    ]
)


X_rdkit = scaler.transform(
    X_rdkit_selected
).astype(
    np.float32
)


assert X_rdkit.shape == (
    1705,
    201
)

assert np.isfinite(
    X_rdkit
).all()


# ============================================================
# 7) BUILD COMBINED 2249
# ============================================================

X_combined = np.concatenate(
    [
        X_morgan.astype(np.float32),
        X_rdkit
    ],
    axis=1
).astype(
    np.float32
)


assert X_combined.shape == (
    1705,
    2249
)

assert np.isfinite(
    X_combined
).all()


print("\n" + "=" * 90)
print("FINAL FEATURE SHAPES")
print("=" * 90)

print(
    "Morgan-only:",
    X_morgan.shape
)

print(
    "RDKit-only:",
    X_rdkit.shape
)

print(
    "Morgan + RDKit:",
    X_combined.shape
)


# ============================================================
# 8) SAVE CACHES
# ============================================================

OUT = Path(
    "/kaggle/working/"
    "FEATURE_ABLATION_TRAIN_ONLY_PREPROCESSING"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


MORGAN_FILE = (
    OUT /
    "molecular_features_morgan_only.npz"
)

RDKIT_FILE = (
    OUT /
    "molecular_features_rdkit_only.npz"
)

COMBINED_FILE = (
    OUT /
    "molecular_features_morgan_rdkit.npz"
)


common = {
    "all_drugs":
        np.asarray(all_drugs),

    "train_drug_indices":
        np.asarray(
            train_drug_indices,
            dtype=np.int64
        ),
}


np.savez_compressed(
    MORGAN_FILE,
    x=X_morgan.astype(np.float32),
    **common
)


np.savez_compressed(
    RDKIT_FILE,
    x=X_rdkit,
    **common
)


np.savez_compressed(
    COMBINED_FILE,
    x=X_combined,
    **common
)


# ============================================================
# 9) SAVE METADATA
# ============================================================

metadata = {

    "experiment":
        "controlled_feature_ablation_train_only",

    "identity_definition":
        "Drug1 and Drug2 are molecular SMILES",

    "total_drugs":
        1705,

    "train_fit_drugs":
        1675,

    "calibration_unseen_drugs":
        14,

    "test_unseen_drugs":
        20,

    "morgan_radius":
        2,

    "morgan_bits":
        2048,

    "morgan_use_chirality":
        False,

    "morgan_use_features":
        False,

    "rdkit_raw_descriptors":
        208,

    "rdkit_raw_nonfinite":
        618,

    "rdkit_retained_train_only":
        201,

    "rdkit_dropped_train_only":
        dropped_names,

    "rdkit_retained_names":
        retained_names,

    "descriptor_selection_rule":
        (
            "Retain descriptor iff Train variance after "
            "Train-median imputation is finite and > 1e-12"
        ),

    "imputation":
        "Train-molecule median",

    "scaling":
        "StandardScaler fitted on Train molecules only",

    "morgan_dim":
        2048,

    "rdkit_dim":
        201,

    "combined_dim":
        2249,
}


META_FILE = (
    OUT /
    "feature_ablation_metadata.json"
)


META_FILE.write_text(
    json.dumps(
        metadata,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 10) RELOAD QA
# ============================================================

for file_path, expected_shape in [

    (
        MORGAN_FILE,
        (1705, 2048)
    ),

    (
        RDKIT_FILE,
        (1705, 201)
    ),

    (
        COMBINED_FILE,
        (1705, 2249)
    ),
]:

    with np.load(
        file_path,
        allow_pickle=True
    ) as data:

        assert data["x"].shape == expected_shape

        assert len(
            data["all_drugs"]
        ) == 1705

        assert len(
            data["train_drug_indices"]
        ) == 1675

        assert np.isfinite(
            data["x"]
        ).all()


# ============================================================
# 11) ZIP
# ============================================================

ZIP_FILE = Path(
    "/kaggle/working/"
    "FEATURE_ABLATION_TRAIN_ONLY_PREPROCESSING_RECOVERED.zip"
)


with zipfile.ZipFile(
    ZIP_FILE,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for p in [
        MORGAN_FILE,
        RDKIT_FILE,
        COMBINED_FILE,
        META_FILE
    ]:

        z.write(
            p,
            arcname=p.name
        )


with zipfile.ZipFile(
    ZIP_FILE,
    "r"
) as z:

    assert z.testzip() is None


# ============================================================
# 12) FINAL
# ============================================================

print("\n" + "=" * 90)
print(" TRAIN-ONLY FEATURE CACHE FULLY RECOVERED")
print("=" * 90)

print(
    "RDKit retained:",
    len(retained_names)
)

print(
    "Dropped:",
    dropped_names
)

print(
    "\nMorgan:",
    X_morgan.shape
)

print(
    "RDKit:",
    X_rdkit.shape
)

print(
    "Combined:",
    X_combined.shape
)

print(
    "\nCombined cache:"
)

print(
    COMBINED_FILE
)

print(
    "\nZIP:"
)

print(
    ZIP_FILE
)

print(
    "\n NOW rerun the Model-selection sensitivity cell."
)